# M3-B1 — Exploration des 3 sources Acerox

> Notebook d'**exploration rapide** — pas d'EDA fouillée, juste assez pour
> remplir l'inventaire de la note d'identification.

Auteur·rice : `<prénom>` — Date : `<date>`

**Règles** :
- Pas de transformation (juste lecture, `info`, `head`, `describe`)
- Une cellule markdown par source — qu'est-ce que tu observes ?
- Trace les **risques** et **questions** qui émergent pour l'`identification_sources.md`

In [4]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path("../data")

## Source 1 — Capteurs IoT (CSV)

In [5]:
df_iot = pd.read_csv(DATA_DIR / "capteurs_iot.csv")
df_iot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      51000 non-null  object 
 1   site           51000 non-null  object 
 2   line_id        51000 non-null  int64  
 3   sensor_id      51000 non-null  object 
 4   temperature_c  51000 non-null  float64
 5   vibration_mms  50251 non-null  float64
 6   debit_uh       51000 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 2.7+ MB


In [6]:
df_iot.head()
df_iot.describe(include="all").transpose().head(10)

print("Valeurs manquantes par colonne:")
print(df_iot.isna().sum())

print("\nTop sites:")
print(df_iot["site"].value_counts())

print("\nTop lignes:")
print(df_iot["line_id"].value_counts().sort_index())

print("\nPériode observée:")
print(df_iot["timestamp"].min(), "->", df_iot["timestamp"].max())

Valeurs manquantes par colonne:
timestamp          0
site               0
line_id            0
sensor_id          0
temperature_c      0
vibration_mms    749
debit_uh           0
dtype: int64

Top sites:
site
Roubaix          20570
Saint-Etienne    20248
Lyon             10182
Name: count, dtype: int64

Top lignes:
line_id
1    21980
2    11846
3    12068
4     5106
Name: count, dtype: int64

Période observée:
2026-04-01T00:00:38 -> 2026-04-29T23:57:53


> **Observations** :
 >
> - Volume : 51 000 lignes, 7 colonnes, environ 3,73 Mo.
 > - Période : avril 2026 (timestamps observés du 2026-04-01 au 2026-04-30).
 > - Qualité observée : 749 valeurs manquantes sur `vibration_mms`, températures extrêmes (jusqu'à 160), plateau fréquent à `vibration_mms = 12.0`.
 > - Risques RGPD : pas de donnée nominative directe, mais exposition possible d'informations industrielles sensibles (site/ligne/capteur/horaires).
 > - Pertinence métier : très forte pour détecter les dérives process en amont des défauts qualité.
 > - Question pour Sébastien : quelle plage de température/vibration est considérée normale par ligne de production ?

## Source 2 — ERP (JSON)

In [7]:
with (DATA_DIR / "erp_export.json").open() as f:
    orders = json.load(f)
df_erp = pd.DataFrame(orders)
df_erp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ordre_id         2000 non-null   int64 
 1   produit_ref      2000 non-null   object
 2   site             2000 non-null   object
 3   line_id          2000 non-null   int64 
 4   date_lancement   2000 non-null   object
 5   date_fin_prevue  2000 non-null   object
 6   statut           2000 non-null   object
 7   ouvrier_id       1891 non-null   object
 8   quantite_kg      2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB


In [8]:
df_erp.head()
df_erp.describe(include="all").transpose().head(10)

print("Valeurs manquantes par colonne:")
print(df_erp.isna().sum())

print("\nRépartition des statuts:")
print(df_erp["statut"].value_counts())

print("\nVérifications clés:")
print("Doublons ordre_id:", df_erp["ordre_id"].duplicated().sum())
print(
    "date_fin_prevue < date_lancement:",
    (pd.to_datetime(df_erp["date_fin_prevue"]) < pd.to_datetime(df_erp["date_lancement"])).sum(),
)

Valeurs manquantes par colonne:
ordre_id             0
produit_ref          0
site                 0
line_id              0
date_lancement       0
date_fin_prevue      0
statut               0
ouvrier_id         109
quantite_kg          0
dtype: int64

Répartition des statuts:
statut
termine     1559
en_cours     197
suspendu     139
annule       105
Name: count, dtype: int64

Vérifications clés:
Doublons ordre_id: 0
date_fin_prevue < date_lancement: 0


> **Observations** :
 >
> - Volume : 2 000 lignes, 9 colonnes, environ 0,56 Mo.
 > - Qualité observée : `ordre_id` sans doublon, cohérence temporelle correcte (`date_fin_prevue >= date_lancement`), 109 valeurs manquantes sur `ouvrier_id`.
 > - Variables métier clés : `statut`, `quantite_kg`, `produit_ref`, `line_id` et les dates de cycle.
 > - ⚠️ Risque RGPD : `ouvrier_id` est une donnée personnelle pseudonymisée, à minimiser et protéger dans les usages analytiques.
 > - Question pour Sébastien : quelle fraîcheur d'export ERP est possible (quotidienne, horaire, quasi temps réel) ?

## Source 3 — Logs machines (texte)

In [9]:
log_path = DATA_DIR / "logs_machines.log"
n_lines = sum(1 for _ in log_path.open())
print(f"Nombre de lignes : {n_lines:,}")
print(f"Taille fichier : {log_path.stat().st_size / 1024:.1f} Ko")

# Aperçu des 5 premières lignes
with log_path.open() as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(line.rstrip())

Nombre de lignes : 30,000
Taille fichier : 1872.8 Ko
[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_changed
[2026-04-01T00:01:07] Saint-Etienne LINE-2 INFO: operator_login
[2026-04-01T00:01:34] Saint-Etienne LINE-3 ERROR: vibration_overlimit sensor=SSAI-L3-T01
[2026-04-01T00:04:18] Roubaix LINE-4 INFO: maintenance_completed
[2026-04-01T00:04:35] Lyon LINE-1 INFO: tooling_loaded


In [11]:
df_logs = pd.read_table(log_path, header=None, names=["raw_log"])

levels = df_logs["raw_log"].str.extract(r"\b(INFO|WARN|ERROR)\b", expand=False)
print("Répartition INFO/WARN/ERROR:")
print(levels.value_counts())

print("\nDoublons de lignes log:", df_logs["raw_log"].duplicated().sum())

hours = pd.to_datetime(
    df_logs["raw_log"].str.extract(r"\[(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})\]", expand=False),
    errors="coerce",
)
print("\nTop 5 minutes les plus chargées:")
print(hours.dt.strftime("%Y-%m-%d %H:%M").value_counts().head(5))

Répartition INFO/WARN/ERROR:
raw_log
INFO     22501
WARN      5758
ERROR     1741
Name: count, dtype: int64

Doublons de lignes log: 1

Top 5 minutes les plus chargées:
raw_log
2026-04-19 19:02    6
2026-04-01 18:44    6
2026-04-06 07:12    5
2026-04-11 13:17    5
2026-04-20 17:55    5
Name: count, dtype: int64


> **Observations** :
 >
> - Format : texte brut non structuré (30 000 lignes), avec niveaux `INFO`, `WARN`, `ERROR`.
 > - Parsing nécessaire : extraction du timestamp, du site, de la ligne et du type d'événement avant usage en modèle.
 > - Qualité observée : 1 doublon détecté ; le reste est exploitable après structuration légère.
 > - RGPD : pas d'identité directe visible, mais des traces liées à l'activité opérateur (`operator_login`, `reason=operator`).
 > - Croisement avec les capteurs IoT : pertinent pour expliquer les pics d'anomalie (ex. vibrations/arrêts d'urgence sur une même ligne).

## Synthèse pour `identification_sources.md`

Remplis le tableau d'inventaire dans `../identification_sources.md` à
partir des observations ci-dessus.